### Input Requirements

> 🚨 **Structure Files**

- `NODE_FILE` *(.xyz, required)*  
  File containing the node fragment (higher-connectivity building unit).  
  **Location:** `0_node/` directory  
  **Constraint:** Exactly one `.xyz` file must be present in this directory.

- `LINKER_FILE` *(.xyz, required)*  
  File containing the linker fragment (lower-connectivity building unit).  
  **Location:** `0_linker/` directory  
  **Constraint:** Exactly one `.xyz` file must be present in this directory.

#### Connection Definition

- `DUMMY_ATOMS` *(required)*  
  Dummy atoms must be placed at all intended connection points (i.e., positions where bonds between node and linker will be formed).  

  - **He** → single bond  
  - **Se** → double bond  

  **Constraint:** Dummy atom types must be consistent with the selected `BOND_TYPE` parameter.

In [1]:
import coflandscaper as cl

/Users/gregorlauter/Documents/PhD/2d-cof-mapper/COF-Landscaper/.venv/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


### General Settings & Options

The following parameters control structure generation and output behavior.

#### Structural Parameters

- `TOPOLOGY` *(str, required)*  
  Defines the underlying network topology.  
  **Allowed values:** `hcb`, `sql`  

- `BOND_TYPE` *(str, required)*  
  Specifies the bond type used to connect node and linker fragments.  
  **Allowed values:** `single`, `double`  
  **Constraint:** Must be consistent with the dummy atoms used in the input structures:  
  - `single` → **He**  
  - `double` → **Se**  
  **Note:** Mismatches may result in incorrect structure generation or runtime errors.

#### Output Parameters

- `COF_NAME` *(str, required)*  
  Unique identifier for the generated structure.  
  All output files and directories will be named using this value.  
  **Constraints:**  
  - Should be unique within the working directory.  
  - Avoid spaces and special characters for compatibility.  

#### Stacking Parameters

- `MODE` *(str, required)*  
  Defines the stacking mode(s) to be generated.  
  **Allowed values:**  
  - `incl` → inclined stacking  
  - `serr` → serrated stacking  
  - `both` → generate both stacking modes  

  **Behavior:**  
  Selecting `both` will generate both configurations sequentially and store them in the corresponding output structure.

In [2]:
TOPOLOGY = "hcb"
BOND_TYPE = "single" 
COF_NAME = "cof-1"
MODE = 'both'

### Single-Layer COF Construction & Pre-Optimization

This step generates the single-layer COF structure and performs a pre-optimization using the MACE model.

In [ ]:
# Construction (Default)
builder = cl.BuildCOF2D()
builder.build(topo=TOPOLOGY, bond_type=BOND_TYPE, cof_name=COF_NAME)

# Pre-Optimization (Default)
preopt = cl.MacePreopt()
preopt.run(cof_name=COF_NAME)

### ILD × ILS Structure Matrix Generation

This step generates stacked COF structures by systematically varying:

- **Interlayer Distance (ILD)** along the $z$-axis  
- **Interlayer Slipping (ILS)** within the plane  

These structures are intended for subsequent single-point energy evaluations and energy landscape analysis.

In [ ]:
# Defaults
matrix = cl.CreateMatrix()
matrix.run(cof_name=COF_NAME, topo=TOPOLOGY, mode=MODE)

### MACE Single-Point Energy Evaluation

This step computes single-point energies for all generated structures using the MACE model, without performing geometry optimization.

This provides a computationally efficient alternative to DFT-based evaluations, enabling rapid screening of stacking configurations

In [ ]:
# Defaults
sp = cl.MaceSP()
sp.run_mode(cof_name=COF_NAME, mode=MODE)

### Potential Energy Landscape (PES)

This step constructs an approximate potential energy landscape by mapping total energies as a function of interlayer distance (ILD) and interlayer slipping (ILS).

The approach assumes that all structures are equivalent aside from their ILD/ILS values, providing a qualitative representation of relative stability across stacking configurations.  
The resulting landscape is used to identify candidate minima for subsequent refinement via full geometry optimization.

In [ ]:
# Defaults
landscape = cl.Landscape()
landscape.run_mode(cof_name=COF_NAME, mode=MODE)

### Additional PES Sampling Points

This allows manual inclusion of additional (ILD, ILS) points in the selection process.
The specified points can appended to the automatically detected minima for each stacking mode.

---

- `EXTRA_SERR` *(list[tuple[float, float]], optional)*  
  Additional points for serrated stacking.  
  Format: `[(ILD, ILS), ...]` in Å  

- `EXTRA_INCL` *(list[tuple[float, float]], optional)*  
  Additional points for inclined stacking.  
  Format: `[(ILD, ILS), ...]` in Å  

In [4]:
EXTRA_SERR = [(3.6, 0.0)]
EXTRA_INCL = [(3.6, 0.0)]

### Structure Selection for Optimization

This step selects candidate structures based on the potential energy landscape and prepares them for downstream optimization.

Structures corresponding to automatically detected minima (and any user-defined ILD/ILS points) are copied into dedicated selection folders.

---

#### Parameters

- `selections_serr`  
  Additional (ILD, ILS) points for serrated stacking to include in the selection.  
  *(list[tuple[float, float]], optional)*

- `selections_incl`  
  Additional (ILD, ILS) points for inclined stacking to include in the selection.  
  *(list[tuple[float, float]], optional)*

- `include_autoselect`  
  If enabled, includes automatically detected minima from the PES.  
  *(bool, optional, default: `False`)*

In [5]:
# Defaults, auto-selected structures
selector = cl.SelectCofs()
selector.run_mode(cof_name=COF_NAME, mode=MODE, include_autoselect=True)


Selected ILD/ILS pairs (Serrated):
 ILD (Å)  ILS (Å)
     3.1      8.7
     3.3      3.0

Selected ILD/ILS pairs (Inclined):
 ILD (Å)  ILS (Å)
     3.1      8.7
     3.4      2.0


In [ ]:
# Defaults, manual selected structures
selector = cl.SelectCofs()
selector.run_mode(cof_name=COF_NAME, mode=MODE, selections_serr=EXTRA_SERR, selections_incl=EXTRA_INCL)

### MACE Geometry Optimization

This step performs geometry optimizations of the selected stacking structures using MACE and retrieves total energies to generate CSV files with relative energies per layer for comparison across stacking configurations.

In [6]:
# Defaults
opt = cl.MaceFullOpt()
opt.run_mode(cof_name=COF_NAME, mode="incl")

       Step     Time          Energy          fmax
LBFGS:    0 14:42:32   -35223.093473        0.252042
LBFGS:    1 14:42:33   -35223.097628        0.116584
LBFGS:    2 14:42:34   -35223.099690        0.093828
LBFGS:    3 14:42:35   -35223.101451        0.070295
LBFGS:    4 14:42:35   -35223.102330        0.054766
LBFGS:    5 14:42:36   -35223.102991        0.052440
LBFGS:    6 14:42:37   -35223.103389        0.043476
LBFGS:    7 14:42:38   -35223.103721        0.033062
LBFGS:    8 14:42:39   -35223.104103        0.034496
LBFGS:    9 14:42:40   -35223.104497        0.035765
LBFGS:   10 14:42:41   -35223.104812        0.038341
LBFGS:   11 14:42:42   -35223.105075        0.030026
LBFGS:   12 14:42:42   -35223.105352        0.022016
LBFGS:   13 14:42:43   -35223.105622        0.027118
LBFGS:   14 14:42:44   -35223.105802        0.028294
LBFGS:   15 14:42:45   -35223.105897        0.026047
LBFGS:   16 14:42:46   -35223.105967        0.016193
LBFGS:   17 14:42:47   -35223.106059        0.01

KeyboardInterrupt: 

### Analysis & Visualization

This step analyzes optimized structures by computing interlayer distance (ILD) and interlayer slipping (ILS), and writing a summary CSV for comparison across stacking configurations.

In addition, structures can be visualized using an interactive viewer with configurable supercell sizes.

In [7]:
# Defaults
analyze = cl.analyze
analyze(cof_name=COF_NAME, mode="serr")

Serrated:
 ILD (Å)  ILS (Å)  Erel (eV)
    3.1      8.7         --
    3.3      3.0         --
    3.6      0.0         --


In [ ]:
# Defaults
visualize = cl.visualize_cof
visualize(cof_name=COF_NAME, mode="incl")

### PXRD Pattern Generation

This step simulates PXRD patterns from optimized CIF structures and writes per-structure `.xy` files.

---

#### Parameters

- `wavelength`  
  X-ray source line used for simulation.  
  *(str, optional, default: `CuKa`)*  
  **Allowed values:** `CuKa`, `MoKa`, `CrKa`, `FeKa`, `CoKa`, `AgKa`  
  Choose this to match your instrument/source.

- `two_theta_range`  
  Angular simulation window in degrees for generated `.xy` data.  
  *(tuple[float, float], optional, default: `(1.5, 60.0)`)*


In [ ]:
# Run: Defaults

pxrd = cl.Pxrd(wavelength="CuKa", two_theta_range=(1.5, 60.0))
pxrd.run(cof_name=COF_NAME, mode=MODE)

### PXRD Plot

This step reads `.xy` files and creates stacked PXRD plots.

#### Parameters

- `xlim`  
  X-axis limits used for plotting in 2-theta degrees.  
  *(tuple[float, float], optional, default: `(1.5, 60.0)`)*

- PXRD patterns are simulated from optimized structures without additional structural modification.  
- Plotting uses stacked representations to facilitate comparison between stacking configurations.

In [ ]:
# Plot: Defaults

pxrd = cl.Pxrd()
pxrd.plot(cof_name=COF_NAME, mode=MODE, xlim=(1.5, 60.0),)